# Solutions — map, filter, lambda, functools & operator

Exercise notebook: `03_pythonic_python/04_map_filter_lambda.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Apply a list of transformations

**Working solution**

In [ ]:
from functools import reduce

def apply_all(value, funcs):
    return reduce(lambda acc, fn: fn(acc), funcs, value)

**Explanation**

`reduce(fn, iterable, initial)` folds left. Unbound methods like `str.strip` are plain functions taking the instance as the first argument.

**Complexity:** O(len(funcs)).

**Common mistakes**

- Forgetting the initial value → `reduce` uses the first function as the accumulator.

## Exercise 2 — 🟡 Medium · Build predicates from keyword arguments

**Working solution**

In [ ]:
_MISSING = object()

def where(**conditions):
    def check(expected, actual):
        if actual is _MISSING:
            return False
        if callable(expected):
            return bool(expected(actual))
        if isinstance(expected, (set, frozenset, list, tuple)):
            return actual in expected
        return actual == expected

    def predicate(record):
        return all(check(exp, record.get(field, _MISSING)) for field, exp in conditions.items())
    return predicate

**Explanation**

`**conditions` collects keyword arguments into a dict; the inner function *closes over* it. Returning a function is how
you build reusable filters — compare with writing a `Predicate<Order>` class hierarchy.

**Complexity:** O(conditions) per record.

**Common mistakes**

- `record[field]` raising `KeyError` inside `filter`.
- Treating strings as collections for membership (`'web' in 'website'` is True!) — only set/list/tuple count.

## Exercise 3 — 🟡 Medium · Sort by field specs with itemgetter

**Working solution**

In [ ]:
from operator import itemgetter

def sort_by(records, *fields):
    result = list(records)
    for field in reversed(fields):
        descending = field.startswith("-")
        result.sort(key=itemgetter(field.lstrip("-")), reverse=descending)
    return result

**Explanation**

Python's sort is **stable**, so sorting by the least significant key first and the most significant last gives a correct
multi-key sort with per-key direction. `itemgetter("x")` is a fast, picklable `lambda r: r["x"]`.

**Complexity:** O(k · n log n).

**Common mistakes**

- Sorting from first field to last — the last sort wins, scrambling the primary order.

## Exercise 4 — 🟠 Hard · compose and pipe

**Working solution**

In [ ]:
from functools import reduce

def pipe(*funcs):
    def run(value):
        return reduce(lambda acc, fn: fn(acc), funcs, value)
    return run

def compose(*funcs):
    return pipe(*reversed(funcs))

**Explanation**

Functions are values: you can store them in tuples, reverse them, and return new functions built from them. `compose` is just `pipe` reversed.

**Complexity:** O(len(funcs)) per call.

**Common mistakes**

- Evaluating the functions eagerly when building (the work must happen when the result is called).

## Challenge — 🔴 Very Hard · Generic group-and-reduce

**Working solution**

In [ ]:
import copy
from operator import itemgetter

def group_reduce(records, key, value, reducer, initial):
    key_fn = itemgetter(key) if isinstance(key, str) else key
    value_fn = itemgetter(value) if isinstance(value, str) else value
    make_initial = initial if callable(initial) else (lambda: copy.deepcopy(initial))
    groups = {}
    for record in records:
        k = key_fn(record)
        acc = groups[k] if k in groups else make_initial()
        groups[k] = reducer(acc, value_fn(record))
    return groups

**Explanation**

This is `reduceByKey` / `groupBy().agg()` in 15 lines. The subtle part is the initial value: reusing one `[]` object for every
group is the mutable-default bug in disguise. A factory (`list`, `dict`, `lambda: 0`) or a deepcopy gives each group its own.

**Complexity:** O(n).

**Common mistakes**

- `groups.setdefault(k, initial)` with `initial=[]` — every group shares the same list.